In [5]:
from google.colab import files

uploaded = files.upload()

Saving WhatsApp Chat with Doomsday (2).txt to WhatsApp Chat with Doomsday (2) (1).txt


In [6]:
from datetime import datetime
import numpy as np


In [16]:
from datetime import datetime

# Assuming only one file was uploaded and we want to process it
# Get the content of the uploaded file
file_content = list(uploaded.values())[0].decode('utf-8')

messages = []

for line in file_content.splitlines():
  line = line.strip()

  if not line:
    continue

  # Remove the invisible character   (Narrow No-Break Space) if present
  line = line.replace('\u202f', '')

  parts = line.split('-', 1)
  if len(parts) != 2:
    continue

  timestamp_text = parts[0].strip()
  message_part = parts[1].strip()

  sender_parts = message_part.split(': ', 1)

  if len(sender_parts) != 2:
    # This line might be a system message or a multi-line message from the previous sender
    # For now, we'll skip it or handle as part of the previous message later if context allows.
    # For this error fix, we'll continue for now.
    continue

  sender = sender_parts[0].strip()
  text = sender_parts[1].strip()

  try:
    # Corrected function name from striptime to strptime
    timestamp = datetime.strptime(timestamp_text, '%d/%m/%y, %I:%M%p')
  except ValueError:
    continue

  messages.append({
    'timestamp': timestamp,
    'sender': sender,
    'text': text
  })

print("Total parsed messages:", len(messages))


Total parsed messages: 11875


In [20]:
# Doomsday - basic stats
total_messages = len(messages)

word_counts = []
media_counts = []
link_counts = []
deleted_counts = []

for m in messages:
  text = m['text']
  word_counts.append(len(text.split()))
  if '<media omitted' in text.lower():
    media_counts.append(1)
  if'https://' in text or 'http://' in text:
    link_counts.append(1)
  if text == '[deleted]':
    deleted_counts.append(1)
word_counts = np.array(word_counts)
total_words = word_counts.sum()
media_counts = np.array(media_counts)
total_media = media_counts.sum()
link_counts = np.array(link_counts)
total_links = link_counts.sum()
deleted_counts = np.array(deleted_counts)
total_deleted = deleted_counts.sum()
print("Total messages :", total_messages)
print("Total words :", total_words)
print("Total media :", total_media)
print("Total links :", total_links)
print("Total deleted :", total_deleted)






Total messages : 11875
Total words : 47635
Total media : 997
Total links : 118
Total deleted : 0.0


In [22]:
def print_bar_chart(labels, values, title, max_width=40):
  print("=" *50)
  print(title)
  print("=" *50)
  for label, value in zip(labels, values):
    bar_length = int(value / max(values) * max_width)
    bar = '#' * bar_length + '-' * (max_width - bar_length)
    print(f"{label}: {value} ({bar})")
  print("=" *50)

In [23]:
#Doomsday - Most active users
names = []
counts = []
for m in messages:
  name = m['sender']
  if name not in names:
    names.append(name)
    counts.append(1)
  else:
    counts[names.index(name)] += 1
#sbse jyada active users ko upar laane ke liye bubble sort
for i in range(len(names)):
  for j in range(i + 1, len(names)):
    if counts[i] < counts[j]:
      names[i], names[j] = names[j], names[i]

top_n = 10
print_bar_chart(names[:top_n], counts[:top_n], "Most active users")


Most active users
+91 93194 59715: 2347 (#########################---------------)
Vikas: 3729 (########################################)
~Prashant: 2594 (###########################-------------)
Jaadu: 134 (#---------------------------------------)
+91 84484 99838: 3026 (################################--------)
Meta AI: 45 (----------------------------------------)


In [25]:
#Doomsday - activity by hour
hour_counts = np.zeros(24)
for m in messages:
  hour_counts[m['timestamp'].hour] += 1

hour_labels = [f"{h:02d}:00" for h in range(24)]
print_bar_chart(hour_labels, hour_counts, "Activity by hour")


Activity by hour
00:00: 279.0 (#######---------------------------------)
01:00: 6.0 (----------------------------------------)
02:00: 0.0 (----------------------------------------)
03:00: 1.0 (----------------------------------------)
04:00: 0.0 (----------------------------------------)
05:00: 0.0 (----------------------------------------)
06:00: 24.0 (----------------------------------------)
07:00: 363.0 (##########------------------------------)
08:00: 402.0 (###########-----------------------------)
09:00: 485.0 (#############---------------------------)
10:00: 531.0 (###############-------------------------)
11:00: 541.0 (###############-------------------------)
12:00: 453.0 (############----------------------------)
13:00: 673.0 (###################---------------------)
14:00: 797.0 (######################------------------)
15:00: 455.0 (############----------------------------)
16:00: 500.0 (##############--------------------------)
17:00: 744.0 (#####################-------

In [36]:
#Doomsday - activities by day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = np.zeros(7)
for m in messages:
  day_counts[m['timestamp'].weekday()] += 1
print_bar_chart(day_names, day_counts, "Activity by day of week")



Activity by day of week
Monday: 1161.0 (##################----------------------)
Tuesday: 1967.0 (##############################----------)
Wednesday: 1673.0 (##########################--------------)
Thursday: 2546.0 (########################################)
Friday: 2267.0 (###################################-----)
Saturday: 1335.0 (####################--------------------)
Sunday: 926.0 (##############--------------------------)


In [28]:
#Doomsday - busiest day
date_list = []
date_counts = []
for m in messages:
  date = m['timestamp'].date()
  if date in date_list:
     idx = date_list.index(date)
     date_counts[idx] += 1
  else:
    date_list.append(date)
    date_counts.append(1)
max_count = max(date_counts)
max_idx = date_counts.index(max_count)
print(f"Busiest day: ", date_list[max_idx], "->", max_count, "messages")



Busiest day:  2025-11-23 -> 299 messages


In [29]:
# Doomsday - Inactive days

all_dates =[m['timestamp'].date() for m in messages]
first_day = min(all_dates)
last_day = max(all_dates)
active_days_set = set(all_dates)
total_days = (last_day - first_day).days + 1
active_days_count = len(active_days_set)
inactive_days_count = total_days - active_days_count
print("First message date :", first_day)
print("Last message date :", last_day)
print("Total days :", total_days)
print("Active days :", active_days_count)
print("Inactive days :", inactive_days_count)

First message date : 2025-11-15
Last message date : 2026-08-15
Total days : 274
Active days : 205
Inactive days : 69


In [35]:
# GroupDNA - Step 9: Most Common Words (English + Hinglish stopwords)
STOPWORDS = ['the', 'is', 'in', 'at', 'of', 'on', 'and', 'a', 'to', 'for',
             'with', 'this', 'that', 'was', 'were', 'be', 'been', 'being',
             'have', 'has', 'had', 'do', 'does', 'did', 'i', 'you', 'he',
             'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
             'my', 'your', 'his', 'its', 'our', 'their', 'am', 'are',
             'but', 'or', 'if', 'so', 'not', 'no', 'yes', 'im', 'ok',
             'okay', 'just', 'very', 'will', 'would', 'can', 'could',
             'media', 'omitted', 'message', 'deleted', 'missed',
             'hai', 'hain', 'ho', 'hoga', 'hogi', 'honge', 'tha', 'thi',
             'ka', 'ki', 'ke', 'ko', 'ku', 'kya', 'kyu', 'kyun', 'kyuki',
             'aur', 'bhi', 'toh', 'to', 'yeh', 'ye', 'wo', 'woh', 'wahi',
             'yahi', 'iske', 'uske', 'iska', 'uska', 'iski', 'uski',
             'mai', 'main', 'mera', 'meri', 'mere', 'tera', 'teri', 'tere',
             'tu', 'tum', 'tumhe', 'tumhara', 'tumhari', 'aap', 'apna', 'apni',
             'hum', 'humara', 'humari', 'unka', 'unki', 'inka', 'inki',
             'nahi', 'nhi', 'kar', 'karo', 'kiya', 'karna',
             'gaya', 'gayi', 'gaye', 'raha', 'rahi', 'rahe', 'liye', 'diya',
             'de', 'do', 'se', 'me', 'pe', 'par', 'ek', 'bas', 'abhi', 'ab',
             'kuch', 'sab', 'sabhi', 'koi', 'kisi', 'jo', 'jis', 'jab', 'tab',
             'lo', 'le', 'liya', 'wala', 'wali', 'wale']


def clean_word(word):
    cleaned = ''
    for ch in word:
        if ch.isalpha():
            cleaned += ch
    return cleaned.lower()


words_list = []
word_counts_list = []

for m in messages:
    for raw in m['text'].split():
        word = clean_word(raw)
        if len(word) <= 2 or word in STOPWORDS:
            continue
        if word in words_list:
            idx = words_list.index(word)
            word_counts_list[idx] += 1
        else:
            words_list.append(word)
            word_counts_list.append(1)

for i in range(len(word_counts_list)):
    for j in range(len(word_counts_list) - 1 - i):
        if word_counts_list[j] < word_counts_list[j + 1]:
            word_counts_list[j], word_counts_list[j + 1] = word_counts_list[j + 1], word_counts_list[j]
            words_list[j], words_list[j + 1] = words_list[j + 1], words_list[j]

top_n = 20
print_bar_chart(words_list[:top_n], word_counts_list[:top_n], f"TOP {top_n} MOST USED WORDS")

TOP 20 MOST USED WORDS
bhai: 763 (########################################)
kal: 377 (###################---------------------)
rha: 274 (##############--------------------------)
prashant: 210 (###########-----------------------------)
aaj: 196 (##########------------------------------)
kon: 193 (##########------------------------------)
vikas: 173 (#########-------------------------------)
college: 161 (########--------------------------------)
anshul: 142 (#######---------------------------------)
dekh: 136 (#######---------------------------------)
gya: 129 (######----------------------------------)
mujhe: 119 (######----------------------------------)
all: 109 (#####-----------------------------------)
yar: 108 (#####-----------------------------------)
fir: 103 (#####-----------------------------------)
kha: 100 (#####-----------------------------------)
baat: 99 (#####-----------------------------------)
haa: 97 (#####-----------------------------------)
hua: 96 (#####----------